# 04 · Zep — 실제 대화로 동작 확인

Zep은 대화를 temporal knowledge graph로 바꿔 기억한다. 실제 LoCoMo 대화를
넣고, 그래프 구축 → bi-temporal 스탬프 → community 생성 → retrieval이
실제로 어떻게 일어나는지 단계별로 확인한다.

```
G_e episode    발화 원문 그대로 (비손실 보관, MENTIONS로 entity에 연결)
G_s semantic   entity 노드 + fact 엣지 — fact마다 시간 축 2개 (valid/invalid)
G_c community  강하게 연결된 entity 묶음 + 요약 (label propagation)
```

Zep의 요점 두 가지를 실물로 본다:

- **지우지 않는다** — 모순되는 새 fact가 오면 옛 fact에 invalid_at을
  찍을 뿐(2단계), 원문 episode와 fact는 그래프에 남는다
- **구조화 비용은 전부 ingest(WRITE)가 낸다** — 발화당 ~22회 LLM 호출.
  대신 answer(READ)는 LLM 호출 없이 검색만 한다 (4단계)

**데이터 — conv-26 세션 3·4** (Caroline & Melanie, 41발화). 노트북 01~03에서
본 그 대화다. 두 세션을 쓰는 이유: 세션 경계가 한 번 있어야 community
rebuild(3단계)를 관찰할 수 있다.

> 이 노트북은 로컬 LM Studio(`qwen3.5-9b-mlx`)와 Neo4j를 호출한다:
> **901호출 / 611K 토큰 / 약 2시간 10분** (M4 Air 실측 — 발화당 평균 ~22호출·~3분).
> 판정 분리 경로였던 직전 실행은 961호출/575K/약 2시간 — 병합(2026-07-11,
> method.py)으로 콜은 6% 줄었지만 병합 프롬프트가 더 무거워 토큰·시간은
> 비슷하다. 병합의 이득은 후보가 쌓인 밀한 그래프에서 커진다.

## 준비 — 부품 조립

용어 하나만: **episode** = 발화 하나를 원문 그대로 담는 노드. 요약본이
아니라 원문이 저장된다 — 그래프가 틀려도 원문으로 돌아갈 수 있다는 게
Zep의 비손실 원칙이다.

아래 셀이 하는 일:

1. conv-26을 로드하고 세션 3·4를 꺼낸다
2. LM Studio에 연결한다 (`llm.calls`/`llm.total_tokens`로 비용 자동 집계)
3. ZepMethod를 만든다 — 생성 시점에 `reset()`이 Neo4j의 모든 노드·엣지를
   지우고 검색 인덱스 6개(BM25 3 + vector 3)를 새로 만든다. 구어로 "DB를
   wipe한다"고 하지만 정식 용어는 아니다 — 개념의 정식 이름은 **실험 격리**:
   대화 10편이 서로 무관하므로 이전 대화의 기억이 새 대화의 QA에 새어드는
   오염을 원천 차단한다 (대화 하나 = 그래프 하나). 운영 서비스라면 삭제
   대신 group_id 파티션으로 격리한다 (우리가 그걸 뺀 이유는 schema.py)
4. 도우미 2개: `ingest_session`(발화를 순서대로 넣기),
   `graph_stats`(그래프의 노드·엣지 수 한 줄 요약)

In [1]:
from memlab.data import load_locomo
from memlab.evaluation import set_f1
from memlab.llm import default_provider
from memlab.methods import Utterance
from memlab.methods.zep import ZepMethod

sample = load_locomo()[0]
s3 = next(s for s in sample.sessions if s.index == 3)
s4 = next(s for s in sample.sessions if s.index == 4)

llm = default_provider()
zep = ZepMethod(llm, sample.speaker_a, sample.speaker_b)
g = zep.graph

def ingest_session(session):
    for turn in session.turns:
        zep.ingest(Utterance(turn.speaker, turn.text,
                             session.date_time, turn.blip_caption))

def graph_stats(title):
    counts = {
        label: g.run(f"MATCH {pattern} RETURN count(x) AS k")[0]["k"]
        for label, pattern in [("episode", "(x:Episodic)"), ("entity", "(x:Entity)"),
                               ("fact", "()-[x:RELATES_TO]->()"),
                               ("community", "(x:Community)")]
    }
    print(f"── {title}: " + " / ".join(f"{k} {v}" for k, v in counts.items()))

print(f"데이터 조각: conv-26 — {sample.speaker_a} & {sample.speaker_b}")
print(f"  세션 3: {len(s3.turns)}발화, t_ref = {s3.date_time}")
print(f"  세션 4: {len(s4.turns)}발화, t_ref = {s4.date_time}  (경계 1회 → rebuild 관찰)")

데이터 조각: conv-26 — Caroline & Melanie
  세션 3: 23발화, t_ref = 7:55 pm on 9 June, 2023
  세션 4: 18발화, t_ref = 10:37 am on 27 June, 2023  (경계 1회 → rebuild 관찰)


### 잠깐 — 코드 셀의 Cypher 읽는 법

그래프 조회는 Cypher로 한다. 패턴을 ASCII 그림으로 그리는 언어다 —
`(노드)-[엣지]->(노드)`가 그래프 모양 그대로. 이 노트북에 나오는 문법은
다섯 개가 전부다:

| 문법 | 뜻 |
|---|---|
| `MATCH (n:Entity)` | Entity 라벨의 노드를 찾아 `n`이라 부른다 |
| `MATCH (a)-[r:RELATES_TO]->(b)` | a에서 b로 가는 엣지 패턴 — `()`는 "아무 노드나", 화살촉 없는 `-`는 방향 무시 |
| `WHERE r.valid_at IS NOT NULL` | 필터 (SQL과 동일) |
| `RETURN n.name AS name` | 출력 컬럼 — `count`·`ORDER BY`·`LIMIT`도 SQL 그대로 |
| `OPTIONAL MATCH` | 패턴이 없어도 행을 버리지 않는다 (LEFT JOIN 격) |

예 — 1단계의 hub 조회 `MATCH (n:Entity)-[e:RELATES_TO]-()`는 "Entity 노드
n에 붙은 모든 엣지 e"라는 그림이고, `count(e)`로 노드별 엣지 수를 세서
`ORDER BY deg DESC LIMIT 8` = 연결 많은 순 8개를 고른다.


## 1. Ingest — 대화가 그래프가 된다

S3의 발화 23개를 순서대로 넣는다. 발화 하나마다 내부에서 일어나는 일:

1. 현재 발화 + 직전 4개(n=4)를 보고 **entity 이름**을 뽑는다 (+ 놓친 것 reflexion)
2. 기존 노드 후보를 검색해 LLM이 **같은 놈인지 판정**(resolution) —
   같으면 병합(summary 결합), 아니면 새 노드
3. 확정된 entity들 사이의 **fact**를 뽑는다 → 같은 pair의 기존 fact와의
   dedup + 모순 용의자 지목(1콜 병합) → t_ref 기준으로 날짜 추출.
   fact 원문이 자구까지 같으면 LLM 없이 즉시 dedup (fast path)

발화당 ~18회 LLM 호출 — 이 셀이 약 1시간으로 노트북에서 가장 길다.

In [2]:
ingest_session(s3)
graph_stats("S3 ingest 후")

print("\n연결 많은 entity top 8 — 재언급이 병합돼 hub가 된다:")
for r in g.run("MATCH (n:Entity)-[e:RELATES_TO]-() "
               "RETURN n.name AS name, count(e) AS deg ORDER BY deg DESC, name LIMIT 8"):
    print(f"  {r['deg']:2d}  {r['name']}")

caroline = g.run("MATCH (n:Entity) WHERE toLower(n.name) CONTAINS 'caroline' "
                 "RETURN n.summary AS s LIMIT 1")
if caroline:
    print(f"\nCaroline 노드의 summary (발화마다 병합을 거친 실물):\n  {caroline[0]['s'][:300]}")
print(f"\n지금까지 {llm.calls}호출 / {llm.total_tokens:,}토큰")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

── S3 ingest 후: episode 23 / entity 22 / fact 61 / community 0

연결 많은 entity top 8 — 재언급이 병합돼 hub가 된다:
  51  Caroline
  42  Melanie
   6  family
   4  transgender journey
   3  LGBTQ community
   2  friends
   2  husband
   2  mentors

Caroline 노드의 summary (발화마다 병합을 거친 실물):
  Caroline, a transgender woman who relocated four years ago and came out three years later, recently celebrated her partner Melanie's wedding day. She sent warm messages complimenting the couple and expressed deep joy over their recent day filled with games, good food, and quality family time. Caroli

지금까지 415호출 / 291,293토큰


출력에서 볼 것:

- **entity 수 < 언급 수**: 화자 두 명이 발화마다 재언급되지만 resolution이
  기존 노드로 병합해 노드는 하나씩이다 — hub의 degree가 그 증거
- **community 0**: 아직 세션 경계를 안 넘었다 — 3단계에서 생긴다

- **415호출 = 발화당 ~18콜**, 산수는 이렇다:

  | 단계 | 콜 수 |
  |---|---|
  | entity 이름 추출 + reflexion | 2 |
  | entity 4~5개 × (summary 1 + resolution 판정 1 + 병합 시 결합 1) | ~10-13 |
  | fact 추출 | 1 |
  | fact 2~3개 × (병합 판정 + 날짜 = 1~2콜, 자구 중복은 0콜) | ~2-5 |

  판정 하나하나가 별도의 LLM 문답이다 — 이게 "구조화 비용을 전부
  WRITE가 낸다"의 실체다. community 유지비는 아직 0인데(첫 세션이라
  G_c가 없음) 3단계부터는 entity당 +2콜이 더 붙는다
- 콜당 평균 ~700토큰 — n=4 창이 담긴 프롬프트 + 짧은 구조화 응답의 평균
  (병합 판정 프롬프트가 지시·예시를 담아 구형 개별 콜보다 무겁다)


## 2. bi-temporal — fact에 시간이 찍힌다

fact마다 시간 축이 둘이다: **valid_at**(사실이 성립한 현실 시간 — 생성 때
t_ref 기준으로 절대화)과 **invalid_at**(모순되는 새 fact가 들어와 무효화된
시간 — 그때가 오면 찍힌다). 이 셀은 LLM 호출 없이 그래프만 조회한다.

In [3]:
print("valid_at이 찍힌 fact (t_ref = 6월 9일 기준으로 절대화된 것들):")
for r in g.run("MATCH ()-[r:RELATES_TO]->() WHERE r.valid_at IS NOT NULL "
               "RETURN r.fact AS fact, r.valid_at AS v ORDER BY r.valid_at LIMIT 8"):
    print(f"  {str(r['v'])[:10]}  {r['fact'][:70]}")

no_date = g.run("MATCH ()-[r:RELATES_TO]->() WHERE r.valid_at IS NULL "
                "RETURN count(r) AS k")[0]["k"]
invalidated = g.run("MATCH ()-[r:RELATES_TO]->() WHERE r.invalid_at IS NOT NULL "
                    "RETURN r.fact AS fact, r.invalid_at AS t")
print(f"\n날짜 없는 fact {no_date}개 — 본문에 시간 정보가 없던 것들")
print(f"invalidated fact {len(invalidated)}개:")
for r in invalidated:
    print(f"  {r['fact'][:65]} → invalid_at {str(r['t'])[:10]}")

valid_at이 찍힌 fact (t_ref = 6월 9일 기준으로 절대화된 것들):
  2018-06-09  Melanie has been married to her husband for 5 years.
  2019-06-09  Caroline has known Melanie for 4 years.
  2023-06-02  Caroline met Melanie last week.

날짜 없는 fact 58개 — 본문에 시간 정보가 없던 것들
invalidated fact 0개:


출력 읽는 법 — 먼저 기준을 상기하자: **이 세션의 t_ref = 2023년 6월 9일**
(setup에서 확인한 세션 3의 날짜)이다.

세 fact 모두 본문의 **상대 표현**이 절대 날짜로 변환된 것이다:

| 본문의 표현 | 절대화 결과 | 산수 |
|---|---|---|
| married for **5 years** | 2018-06-09 | 2023-06-09 − 5년 |
| known for **4 years** | 2019-06-09 | 2023-06-09 − 4년 |
| **last week** | 2023-06-02 | 2023-06-09 − 7일 |

앞 두 개의 월·일이 똑같이 "06-09"인 게 그 증거다 — 기준일에서 연 단위를
뺀 것. 5년 전 결혼이라는 **과거의 사실을 2023년 대화에서 알게 된** 셈인데,
이걸 구분하는 게 bi-temporal의 요지다. 시간 축이 둘이다:

| 축 | 필드 | 묻는 것 |
|---|---|---|
| **T — 현실 시간** | valid_at / invalid_at | 사실이 세상에서 **언제부터 언제까지 참**이었나 |
| **T′ — 기록 시간** | created_at / expired_at | 시스템이 그 사실을 **언제 알았고 언제 폐기**했나 |

위 출력의 날짜는 전부 T축이다 (Melanie의 결혼: T = 2018년~, T′ = 2023-06-09
ingest 시각). QA("결혼 몇 년 차?")와 모순 판정이 쓰는 축도 T다.

- invalid_at은 T축의 "끝"이다 — **invalidated 0개** = 이 조각의 fact들은
  전부 아직 유효한 열린 구간이고, 4단계 context에서 "(... - present)"로
  보이는 게 그것이다. 모순이 있어야 닫히는데 근황 공유 대화에선 드물다 —
  0개도 실측이다 (무효화 발동은 전량 런의 긴 대화에서 본다)
- 날짜 없는 fact 52개 — "가족이 힘이 된다" 같은 무시간 서술은 valid_at이
  비는 게 맞다 (temporal 추출 프롬프트의 지침)


## 3. 세션 경계 — community가 생긴다

S4를 이어서 넣는다. 감지 장치는 단순하다: LoCoMo의 발화에는 **자기 세션의
날짜 문자열**이 붙어 있고(세션 3의 발화 23개는 전부 "7:55 pm on 9 June,
2023"), method는 직전 발화의 문자열을 기억해 둔다. S4의 첫 발화가 들고 온
문자열("10:37 am on 27 June, 2023")이 직전 것과 **다르다** — 이 문자열 비교
한 줄이 "세션 경계 감지"의 전부다. 별도 신호도, 세션 번호도 없다.

경계를 만나면 그 발화를 처리하기 **전에** G_c full rebuild가 돈다:
label propagation이 강하게 연결된 entity를 묶고, 묶음마다 member
summary들을 map-reduce로 접어 community summary를 만든다. 이게 논문의
"주기적 refresh"를 LoCoMo의 자연 주기(세션 = 며칠 간격의 대화 재개)에
맞춘 것 — 한 회차가 끝날 때마다 기억을 한 번 정리하는 셈이다.
(재구축 장치 자체는 논문·원본에 있지만 **주기를 세션으로 정한 건 우리의
해석**이다 — 논문은 주기를 침묵하고, 원본은 호출 시점을 호출자에 맡긴다.
근거 기록은 communities.py docstring.)

세션이 진행되는 동안은 발화가 건드린 entity마다 **dynamic extension** —
무소속이면 이웃 다수결로 기존 community에 합류하고, 그 community의
summary가 갱신된다.


In [4]:
ingest_session(s4)
graph_stats("S4 ingest 후")

print("\ncommunity — member 수와 name (label propagation의 결과):")
for r in g.run("MATCH (c:Community) OPTIONAL MATCH (c)-[:HAS_MEMBER]->(m) "
               "RETURN c.name AS name, count(m) AS k ORDER BY k DESC, name"):
    print(f"  {r['k']:2d}명  {r['name'][:70]}")

biggest = g.run("MATCH (c:Community)-[:HAS_MEMBER]->(m) WITH c, count(m) AS k "
                "ORDER BY k DESC LIMIT 1 RETURN c.summary AS s")[0]["s"]
print(f"\n최대 community의 summary (map-reduce 요약 실물):\n  {biggest[:400]}")
print(f"\n지금까지 {llm.calls}호출 / {llm.total_tokens:,}토큰")

── S4 ingest 후: episode 41 / entity 37 / fact 95 / community 11

community — member 수와 name (label propagation의 결과):
  20명  This summary details Caroline's personal journey from a transgender wo
   3명  This summary contrasts Melanie's positive family camping experience an
   1명  This summary captures Caroline's reflections on her supportive transit
   1명  This summary captures a dialogue between Caroline and Melanie regardin
   1명  This summary captures a personal account of Caroline's transgender jou
   1명  This summary describes a specific photograph of Melanie and her daught
   1명  This summary explains that a photo of Caroline's family in a yard was 
   1명  This summary highlights the shared importance of family as a source of
   1명  This summary outlines Caroline's motivation to share her personal gend
   1명  This summary outlines Caroline's school presentation on gender identit
   1명  This summary outlines how Caroline's personal resilience and support n

최대 community의 summary (m

출력에서 볼 것:

- **큰 community 하나 + 싱글톤들** — 대화가 두 사람 근황이라 화자 중심으로
  뭉치고, 스치듯 언급된 entity는 혼자 남는다
- rebuild 시점은 S4 시작이라 요약의 뼈대는 S3 내용이고, S4 내용은
  dynamic extension이 얹었다 — 다음 세션 경계(스모크에선 S5 시작)에서
  다시 전체 재구축된다

## 4. Retrieval — 검색 셋 + RRF + context

`answer()`가 쓰는 f(α) = χ(ρ(φ(α)))를 열어본다: 질문 하나를 **BM25·cosine·BFS** 세 방법으로 검색하고(φ), 랭킹 셋을 RRF로 융합하고(ρ),
FACTS/ENTITIES/COMMUNITIES context 문자열로 조립한다(χ). LLM 호출은 없다 —
retrieval은 READ 전용이고 임베딩 1회만 쓴다.

문제는 5단계 시험지에서 가장 어려운 것(03이 0.00을 받은 Sweden multi-hop)을
미리 쓴다.

In [5]:
from memlab.embedding import embed

question = sample.qa[11].question
vec = [float(x) for x in embed(question)]
mr = zep.retrieval  # 관찰을 위해 sub-search를 직접 부른다

bm25 = mr._edge_bm25(question, 20)
cos = mr._edge_cosine(vec, 20)
origins = list(dict.fromkeys(e.source_uuid for e in bm25 + cos))
bfs = mr._edge_bfs(origins, 20)

print(f"Q: {question}\n")
print(f"φ fact 검색 — BM25 {len(bm25)}건 / cosine {len(cos)}건 / BFS {len(bfs)}건")
print(f"  BM25 1위:   {bm25[0].fact[:70] if bm25 else '(없음)'}")
print(f"  cosine 1위:   {cos[0].fact[:70] if cos else '(없음)'}")

context = mr.retrieve(question)
print(f"\nχ가 조립한 context (~{len(context) // 4}토큰):\n")
print(context)

Q: Where did Caroline move from 4 years ago?

φ fact 검색 — BM25 20건 / cosine 20건 / BFS 20건
  BM25 1위:   Caroline has known Melanie for 4 years.
  cosine 1위:   Caroline has had a lot of life events happen recently.



χ가 조립한 context (~3759토큰):

FACTS, ENTITIES and COMMUNITIES represent relevant context to the current conversation.

These are the most relevant facts and their valid date ranges. If the fact is about an event, the event takes place during this time.
format: FACT (Date range: from - to)
<FACTS>
Caroline has known Melanie for 4 years. (Date range: 2019-06-09 19:55:00 - present)
Caroline is motivated by her mentors. (Date range: unknown - present)
Caroline has had a lot of life events happen recently. (Date range: unknown - present)
Caroline wishes Melanie many happy years together. (Date range: unknown - present)
Mentors give strength to Caroline. (Date range: unknown - present)
Family gives strength to Caroline. (Date range: unknown - present)
The grandma is from Sweden. (Date range: unknown - present)
Caroline is motivated by her family. (Date range: unknown - present)
Caroline met Melanie last week. (Date range: 2023-06-02 00:00:00 - present)
Caroline and Melanie met up last week. (D

출력에서 볼 것:

- 세 검색이 **다른 것을 잡는다** — BM25는 단어 겹침, cosine은 의미,
  BFS는 검색에 걸린 entity의 이웃 (질문에 없는 fact가 딸려온다)
- context의 FACTS 상단 = 여러 랭킹에 동시에 걸린 fact (RRF의 효과)
- fact마다 `(Date range: ...)` — 2단계의 bi-temporal 스탬프가 답변 LLM에게
  그대로 노출된다
- 이 질문의 결정적 단서("from Sweden")는 FACTS가 아니라 ENTITIES·
  COMMUNITIES의 summary에 실려 온다 — 세 섹션이 상호 보완하는 장면
- "20건"인데 FACTS가 10줄인 이유: 각 검색은 최종 limit(10)의 **2배씩 후보를
  모아 온다** (원본의 2×limit 그대로) — RRF가 겹침을 볼 재료를 넉넉히 받고,
  자르는 건 융합 후 한 번이다. 이 READ 경로 전체에 LLM 호출은 없다


## 5. QA — 기억의 상태가 답을 가른다

상태가 다른 다섯 문제를 내고 set_f1(1.0 만점)로 채점한다. 각 문제가
Zep의 어느 장치를 시험하는지:

| 문제 | 시험하는 장치 |
|---|---|
| 연설 시기 | valid_at — "last week"의 t_ref 절대화 (2단계) |
| 4년 전 어디서 이사왔나 | fact 원문 보존 — 요약이 아니라 fact 단위 저장 |
| Mel 부부 결혼 몇 년 차 | 지우지 않는 구조 — 오래된 fact도 그래프에 남는다 |
| 캠핑에서 뭐 했나 | 평범한 retrieval 경로 |
| 어떤 공간을 만들고 싶나 | 마지막 세션 끝자락의 fact |

(03과 같은 문제셋이다 — 다른 메소드가 같은 조각에서 어땠는지 궁금하면
03의 5단계와 나란히 놓고 볼 수 있다. 단 03은 Groq 기록이라 모델이 다르다.)

In [6]:
exam = [
    (8, "valid_at — 연설 시기"),
    (11, "fact 보존 — 4년 전 이사 (multi-hop)"),
    (90, "비삭제 구조 — 결혼 몇 년 차"),
    (95, "평범한 retrieval — 캠핑"),
    (100, "마지막 화제 — 만들고 싶은 공간"),
]
for index, note in exam:
    qa = sample.qa[index]
    prediction = zep.answer(qa.question)
    print(f"[{note}]")
    print(f"Q: {qa.question}")
    print(f"  정답: {qa.answer}")
    print(f"  예측: {prediction}")
    print(f"  set_f1 = {set_f1(prediction, qa.answer):.2f}\n")

print(f"총 비용: {llm.calls}호출 / {llm.total_tokens:,}토큰")


[valid_at — 연설 시기]
Q: When did Caroline give a speech at a school?
  정답: The week before 9 June 2023
  예측: last week
  set_f1 = 0.25



[fact 보존 — 4년 전 이사 (multi-hop)]
Q: Where did Caroline move from 4 years ago?
  정답: Sweden
  예측: Sweden
  set_f1 = 1.00



[비삭제 구조 — 결혼 몇 년 차]
Q: How long have Mel and her husband been married?
  정답: Mel and her husband have been married for 5 years.
  예측: 5 years
  set_f1 = 0.33



[평범한 retrieval — 캠핑]
Q: What did Melanie and her family do while camping?
  정답: explored nature, roasted marshmallows, and went on a hike
  예측: roasting marshmallows around a campfire
  set_f1 = 0.29



[마지막 화제 — 만들고 싶은 공간]
Q: What kind of place does Caroline want to create for people?
  정답: a safe and inviting place for people to grow
  예측: safe, inviting environments where individuals can achieve self-acceptance and personal growth
  set_f1 = 0.29

총 비용: 901호출 / 610,699토큰


출력에서 볼 것:

- **Sweden 1.00** — 03이 0.00이던 문제. 요약이 "home country"로 뭉갰던
  정보가 Zep에선 fact와 summary에 원문 그대로 남아 있었다 — 구조 차이가
  점수로 실증된 장면
- **결혼 문제의 0.17** — 예측 "five years"는 내용상 정답이다. 03의 0.00은
  기억이 없어서였고, 이번 낮은 점수는 긴 정답 문장과의 토큰 겹침이 작은
  set_f1의 채점 특성이다 (02 참고)
- **연설 시기 0.25** — context에 date range가 있는데도 답변 모델이 "last
  week"이라고 답했다. 절대화된 날짜를 답에 쓰는 건 retrieval이 아니라
  답변 LLM의 몫 — 로컬 소형 모델의 한계가 드러나는 지점


## 정리

- **Ingest**: 발화가 episode로 남고 entity·fact가 그래프에 쌓인다 —
  지우는 단계가 없다. 재언급은 resolution이 병합한다
- **bi-temporal**: fact의 시작은 생성 때 t_ref 기준으로, 끝은 모순 fact가
  올 때 찍힌다 — 삭제 대신 무효화
- **community**: 세션 경계마다 label propagation으로 재구축, 세션 중엔
  이웃 다수결로 합류
- **retrieval**: 검색 셋(fulltext·cosine·BFS) → RRF → date range가 달린
  FACTS/ENTITIES/COMMUNITIES context

**다음 — 전량 런**: 대화 10편을 러너로 돌려 baseline 수치를 얻는다
(`uv run memlab-run --method zep --run-id zep-baseline`, 체크포인트 = 대화 단위).
러너의 zep 설정은 community 유지보수를 끈다 — QA가 전부 ingest 뒤라
동작 차이 없이 비용만 줄기 때문 (근거: ZepConfig.update_communities).